In [1]:
#import
import pandas as pd
import numpy as np
import glob
import os

In [4]:
print(os.getcwd())
os.chdir('c:/Users/tprop/Documents/big-data-bowl/Analytics/iaa-2026-big-data-bowl')

c:\Users\tprop\Documents\big-data-bowl\Analytics\iaa-2026-big-data-bowl\notebooks


In [6]:
#data load

# folder containing the CSVs
folder_path = "./data/train"

# find all CSVs starting with "input" or "output"
input_files = glob.glob(os.path.join(folder_path, "input*.csv"))
output_files = glob.glob(os.path.join(folder_path, "output*.csv"))
supplementary = pd.read_csv('./data/supplementary_data.csv')

# load and combine into one DataFrame
input = pd.concat((pd.read_csv(f) for f in input_files), ignore_index=True)
output = pd.concat((pd.read_csv(f) for f in output_files), ignore_index=True)


C:\Users\tprop\AppData\Local\Temp\ipykernel_29428\3066230248.py:9: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  supplementary = pd.read_csv('./data/supplementary_data.csv')


In [22]:
#sample play
sample = input[(input['game_id']==2023090700) & (input['play_id']==101)]

In [24]:
target_rec = sample[sample['player_role'] == "Targeted Receiver"].copy()
other_players = sample[sample['player_role'] != "Targeted Receiver"].copy()

In [ ]:
# Rename columns for clarity after merge
target_rec = target_rec.rename(columns={'x': 'target_x', 'y': 'target_y', 
                                       'nfl_id': 'target_id'})
other_players = other_players.rename(columns={'x': 'other_x', 'y': 'other_y', 
                                     'nfl_id': 'other_id'})
    
# Cross join within each frame
merged = target_rec.merge(other_players, on='frame_id', suffixes=('_target', '_other'))
    
# Calculate distances
merged['distance'] = np.sqrt(
    (merged['target_x'] - merged['other_x'])**2 + 
    (merged['target_y'] - merged['other_y'])**2
)
    
# Find minimum distance per frame (and target if multiple)
idx = merged.groupby(['frame_id', 'target_id'])['distance'].idxmin()
closest = merged.loc[idx].reset_index(drop=True)

Need to match player IDs (nfl_id) to output to get join, get roles and continue to track closest players in output, also need to filter to only track closes DEFENSIVE player.

#### Route Types

In [8]:
route_types = supplementary['route_of_targeted_receiver'].value_counts()
route_types

route_of_targeted_receiver
HITCH     3383
OUT       2886
FLAT      2490
CROSS     1957
GO        1776
IN        1408
SLANT     1316
POST       978
ANGLE      704
CORNER     642
SCREEN     369
WHEEL       96
Name: count, dtype: int64

Positions if receivers won
* Hitch - receiver in front
* Out - reciever in front
* Flat - reciever in front?
* Cross - receiver in front/also horizontal distance
* Go - reciver behind
* In - receiver in front/also horizontal distance
* Slant - receiver in front/also horizontal distance
* Post - reciver behind
* Angle
* Corner
* Screen - ?
* Wheel - reciver behind